# 05 - Paired headline generation: full run (one click)

**One run does everything:**
1. **Judge contest** on 400 real training headlines with known labels: SmolLM3-3B vs cross-voting by the generator models. The winner is the judge if it reaches 0.75 balanced accuracy; otherwise no judge filter.
2. **Generation**: a sarcastic and a serious headline for every training headline (22,800), split equally over Qwen, Gemma and Mistral; both sides of a pair from the same model.
3. **Checks + judging** with the winner only.
4. **Frozen dataset**: `pairs_train.csv` (what Pass 4 trains on) with `manifest_pairs.json` (SHA-256, scores, counts).

**How to run:** right-hand panel: GPU T4 x2, Internet on, input `gainsplit-processed`, secret `HF_TOKEN`. Then **Save Version > Save & Run All (Commit)** and close the tab. Expected about 7-9 hours (estimated from the pilot), under Kaggle's 12-hour limit. When it is done, download `pairgen.zip` from the version's Output.

**If it is cut short:** chunks of 500 headlines are saved as it goes, and generation stops by itself after `MAX_GEN_HOURS` so the output is kept. The last cell then says "not finished". Add this version's output as input (Add Input > Your Work) and commit again: it continues where it stopped and finishes.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_pairgen.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "tests failed"

## Settings (no need to change anything)

In [ ]:
from pathlib import Path
N_SEEDS = None            # None = whole training split
SEED = 2026
CHUNK = 500
MAX_GEN_HOURS = 9.5       # stop generating after this, so the run ends inside Kaggle's 12 hours
MODELS = {
    "qwen":    {"id": "Qwen/Qwen2.5-7B-Instruct",           "extra": {}},
    "gemma":   {"id": "google/gemma-2-9b-it",               "extra": {"attn_implementation": "eager"}},
    "mistral": {"id": "mistralai/Mistral-7B-Instruct-v0.3", "extra": {}},
}
SMOL = {"id": "HuggingFaceTB/SmolLM3-3B", "extra": {}}
GEN = {"max_new_tokens": 120, "temperature": 0.8, "top_p": 0.95, "batch_size": 32}
JUDGE_BATCH = 64
N_REAL_CALIB = 400
JUDGE_MIN = 0.75
OUT = Path("/kaggle/working/pairgen")
RAW = OUT / "raw_chunks"
RAW.mkdir(parents=True, exist_ok=True)
INPUT = Path("/kaggle/input")

In [ ]:
import json, hashlib, gc, shutil
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from gainsplit.data import find_data_dir
from gainsplit import pairgen as pg

DATA_DIR = find_data_dir(str(INPUT))
train = pd.read_csv(DATA_DIR / "headlines_train.csv", keep_default_na=False)
plan = pg.plan_seeds(train, list(MODELS), SEED, N_SEEDS)
plan_hash = hashlib.sha256(pd.util.hash_pandas_object(plan, index=False).values.tobytes()).hexdigest()[:16]
for f in list(INPUT.rglob("pairgen/plan_fingerprint.txt")):
    assert f.read_text().strip() == plan_hash, f"seed plan differs from {f}"
(OUT / "plan_fingerprint.txt").write_text(plan_hash)
plan.to_csv(OUT / "seed_plan.csv", index=False)
print("seeds:", len(plan), "| plan:", plan_hash)

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
HF_TOKEN = get_secret("HF_TOKEN")
print("HF_TOKEN found:", HF_TOKEN is not None, "| GPUs:", torch.cuda.device_count())

def load(spec):
    tok = AutoTokenizer.from_pretrained(spec["id"], token=HF_TOKEN)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(spec["id"], token=HF_TOKEN, torch_dtype=torch.float16,
                                                 device_map="auto", **spec["extra"])
    model.eval()
    return tok, model

def run_prompts(tok, model, prompts, batch_size, max_new_tokens, sample):
    outs = []
    for i in range(0, len(prompts), batch_size):
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
                 for p in prompts[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        kw = dict(do_sample=True, temperature=GEN["temperature"], top_p=GEN["top_p"]) if sample else dict(do_sample=False)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def judge_texts(spec, texts):
    tok, model = load(spec)
    answers = run_prompts(tok, model, [pg.judge_prompt(t) for t in texts], JUDGE_BATCH, 16, sample=False)
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return [pg.parse_judge(a) for a in answers]

## 1. Judge contest (about 10-15 minutes; reused if a previous run already did it)

In [ ]:
contest_files = [OUT / "judge_contest.json"] + list(INPUT.rglob("pairgen/judge_contest.json"))
found = [f for f in contest_files if f.exists()]
if found:
    contest = json.loads(found[0].read_text())
    print("contest reused from", found[0])
else:
    calib = pg.pick_seeds(train, N_REAL_CALIB, SEED + 1)
    y = calib["seed_label"].tolist()
    preds = {name: judge_texts(spec, calib["seed_text"].tolist()) for name, spec in {"smollm3": SMOL, **MODELS}.items()}
    detail = {"smollm3": pg.balanced_accuracy(y, preds["smollm3"])}
    for m in MODELS:
        others = [g for g in MODELS if g != m]
        detail[f"cross without {m}"] = pg.balanced_accuracy(y, [pg.cross_verdict(v) for v in zip(*(preds[g] for g in others))])
    scores = {"smollm3": detail["smollm3"], "cross": float(np.mean([detail[f"cross without {m}"] for m in MODELS]))}
    contest = {"scores": {k: round(v, 3) for k, v in detail.items()}, "summary": {k: round(v, 3) for k, v in scores.items()},
               "min_score": JUDGE_MIN, "chosen": pg.pick_judge(scores, JUDGE_MIN), "real_headlines": N_REAL_CALIB}
    (OUT / "judge_contest.json").write_text(json.dumps(contest, indent=2))
print(json.dumps(contest, indent=2))
JUDGE = contest["chosen"]

## 2. Generation (most of the run; saved every 500 headlines)

In [ ]:
for p in pg.find_chunks(INPUT):
    dest = RAW / p.parent.name / p.name
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, dest)

out_of_time = False
for name, spec in MODELS.items():
    chunks = pg.model_chunks(plan, name, CHUNK)
    todo = [k for k in range(len(chunks)) if not (RAW / name / pg.chunk_name(k)).exists()]
    print(f"{name}: {len(todo)} of {len(chunks)} chunks to do")
    out_of_time = out_of_time or (time.time() - T_START) / 3600 > MAX_GEN_HOURS
    if not todo or out_of_time:
        continue
    tok, model = load(spec)
    (RAW / name).mkdir(parents=True, exist_ok=True)
    for k in todo:
        if (time.time() - T_START) / 3600 > MAX_GEN_HOURS:
            out_of_time = True; break
        part = chunks[k].copy()
        torch.manual_seed(SEED * 10_000 + k)
        t1 = time.time()
        part["raw"] = run_prompts(tok, model, [pg.build_prompt(t) for t in part["seed_text"]],
                                  GEN["batch_size"], GEN["max_new_tokens"], sample=True)
        part.to_csv(RAW / name / pg.chunk_name(k), index=False)
        print(f"  {name} chunk {k}: {time.time() - t1:.0f} s | run time {(time.time() - T_START) / 3600:.1f} h")
    del model, tok; gc.collect(); torch.cuda.empty_cache()

raw = pg.collect_raw(pg.find_chunks(OUT))
raw = raw[raw["seed_id"].isin(plan["seed_id"])]
FINISHED = len(raw) == len(plan)
print("generated:", len(raw), "of", len(plan), "| finished:", FINISHED)

## 3. Checks, judging with the winner, frozen dataset

In [ ]:
if not FINISHED:
    print("NOT FINISHED: add this version's output as input and commit again; it continues from here.")
else:
    checks = pd.DataFrame([pg.check_pair(s, r) for s, r in zip(raw["seed_text"], raw["raw"])])
    pairs = pd.concat([raw.reset_index(drop=True), checks], axis=1)
    items = pg.long_items(pairs[pairs["ok"]])

    if JUDGE == "smollm3":
        items["verdict"] = judge_texts(SMOL, items["text"].tolist())
    elif JUDGE == "cross":
        votes = {}
        for g, spec in MODELS.items():
            sub = items[items["model"] != g]
            votes[g] = dict(zip(sub["item_id"], judge_texts(spec, sub["text"].tolist())))
        items["verdict"] = [pg.cross_verdict([votes[g][i] for g in MODELS if g != m])
                            for i, m in zip(items["item_id"], items["model"])]
    else:
        items["verdict"] = None
    items["agrees"] = (items["verdict"] == items["intended"]) if JUDGE else True
    items.to_csv(OUT / "judged_items.csv", index=False)

    both = items.groupby("seed_id")["agrees"].agg(lambda s: bool(len(s) == 2 and s.all()))
    pairs["kept"] = pairs["ok"] & pairs["seed_id"].map(both).fillna(False).astype(bool)
    pairs.to_csv(OUT / "pairs_all.csv", index=False)
    rows_train = pg.training_rows(pairs)
    rows_train.to_csv(OUT / "pairs_train.csv", index=False)

    cues = json.loads((DATA_DIR / "cue_words.json").read_text())
    sc = [w for w, _ in cues["sarcastic"]]
    kept = pairs[pairs["kept"]]
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    manifest = {
        "commit": COMMIT, "plan_fingerprint": plan_hash, "seed": SEED, "n_seeds": len(plan), "models": MODELS,
        "prompt_version": pg.PROMPT_VERSION, "gen": GEN, "judge_contest": contest,
        "counts": {"pairs_ok": int(pairs["ok"].sum()), "pairs_kept": int(pairs["kept"].sum()),
                   "training_rows": len(rows_train),
                   "ok_by_model": pairs.groupby("model")["ok"].mean().round(3).to_dict(),
                   "kept_by_model": pairs.groupby("model")["kept"].mean().round(3).to_dict()},
        "sarcastic_cue_rate": {"kept_sarcastic_side": round(pg.cue_rate(kept["sarcastic"], sc), 3),
                               "kept_serious_side": round(pg.cue_rate(kept["serious"], sc), 3),
                               "real_sarcastic": round(pg.cue_rate(train.loc[train["label"] == 1, "text"], sc), 3),
                               "real_serious": round(pg.cue_rate(train.loc[train["label"] == 0, "text"], sc), 3)},
        "examples": kept.sample(min(8, len(kept)), random_state=SEED)[["model", "seed_text", "sarcastic", "serious"]].to_dict("records"),
        "hours": round((time.time() - T_START) / 3600, 2),
        "sha256": {f: sha(OUT / f) for f in ["pairs_all.csv", "pairs_train.csv", "judged_items.csv", "seed_plan.csv"]},
    }
    (OUT / "manifest_pairs.json").write_text(json.dumps(manifest, indent=2, default=str))
    print(json.dumps({k: v for k, v in manifest.items() if k != "sha256"}, indent=2, default=str))
    print("DONE")

In [ ]:
print(shutil.make_archive("/kaggle/working/pairgen", "zip", OUT))